# Capstone: boosting and explainable AI

Two questions, one run (CPU):

1. **Is the random forest the best tree model for habits?** LightGBM and CatBoost, with the same 15
   inputs, settings fixed in advance, and the same threshold rule, next to the baseline, logistic
   regression and the forest, on REFIT, UK-DALE and CASAS (`scripts/eval_refit.py --models ...`).
2. **Explainable AI (SHAP).** Which inputs drive each habit model's predictions
   (`scripts/explain_habits.py`, for the forest, LightGBM and CatBoost), and what drives the
   gradient-boosting load forecaster (`scripts/explain_forecast.py`, REFIT and UK-DALE), and why the
   forest guard restored power on real test nights, for rescues and false alarms
   (`scripts/explain_guard.py`, REFIT and UK-DALE).

**Settings:** Accelerator = **None** (CPU), Internet = **On** (CASAS from Zenodo, London's solar from
PVGIS), dataset `capstone-bundle` (latest version) under Input. Save Version > Save & Run All (Commit).
Phone notifications as before (secret `NTFY_URL`). Expect a few hours; each step logs to `logs/`.

**A new notebook copies none of these settings.**

In [ ]:
import glob, os, shutil, subprocess
# os.environ['NTFY_URL'] = 'https://ntfy.sh/<your-topic>'   # used only if the NTFY_URL secret is missing
found = glob.glob('/kaggle/input/**/habits', recursive=True)
assert found, 'No bundle: add the capstone-bundle dataset with Add Input (right-hand panel), then rerun'
bundle = os.path.dirname(found[0])
shutil.copytree(bundle, '/kaggle/working/capstone', dirs_exist_ok=True)
os.chdir('/kaggle/working/capstone')
os.makedirs('logs', exist_ok=True)
print('bundle from', bundle)
!pip install -q paho-mqtt
import torch, sklearn, pandas, sys
print('python', sys.version.split()[0], '| torch', torch.__version__, '| sklearn', sklearn.__version__,
      '| pandas', pandas.__version__)
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'none (check the Accelerator setting)')
import lightgbm, catboost, shap
print('lightgbm', lightgbm.__version__, '| catboost', catboost.__version__, '| shap', shap.__version__)
sys.path.insert(0, 'scripts')
from kaggle_steps import ADDRESS, notify, run_step
print('ntfy:', 'on' if ADDRESS else 'off (no NTFY_URL secret or first-line topic)')
notify('Boosting + XAI: started', 'CASAS download, then the runs', 'default', 'rocket')

In [ ]:
!mkdir -p /kaggle/temp && wget -q -c -O /kaggle/temp/casas.zip "https://zenodo.org/records/15708568/files/labeled_data.zip?download=1"
!ls -l /kaggle/temp/casas.zip   # should be 236037656 bytes
ok = run_step('CASAS prepare', 'CAPSTONE_DATASET=casas python -u scripts/prepare_casas.py /kaggle/temp/casas.zip',
              'logs/casas_prepare.log', summary_lines=3)
casas_ok = ok  # without CASAS, its steps are skipped (REFIT and UK-DALE still run)

In [ ]:
DATASETS = ['refit', 'ukdale', 'casas']   # to rerun only CASAS: ['casas']
MODELS = ['random_forest', 'lightgbm', 'catboost']
ok &= run_step('Tests', 'python -m pytest -q tests/test_features.py tests/test_habits.py tests/test_explain.py', 'logs/tests.log', summary_lines=2)
for ds in DATASETS:
    if ds == 'casas' and not casas_ok:
        continue
    env = {'CAPSTONE_DATASET': ds}
    ok &= run_step(f'{ds} habit models + boosting', 'python -u scripts/eval_refit.py --models lightgbm,catboost',
                   f'logs/{ds}_habit_boosting.log', env=env, summary_lines=30)
    for m in MODELS:
        ok &= run_step(f'{ds} SHAP {m}', f'python -u scripts/explain_habits.py --model {m}',
                       f'logs/{ds}_shap_{m}.log', env=env, summary_lines=25)
    if ds != 'casas':  # CASAS has no electricity load to forecast, nor standby cuts
        ok &= run_step(f'{ds} SHAP forecaster', 'python -u scripts/explain_forecast.py',
                       f'logs/{ds}_shap_forecast.log', env=env, summary_lines=15)
        ok &= run_step(f'{ds} guard reasons', 'python -u scripts/explain_guard.py',
                       f'logs/{ds}_guard_reasons.log', env=env, summary_lines=40)

In [ ]:
import zipfile, pathlib
keep = ('habit_models', 'habit_shap', 'forecast_shap', 'guard_reasons')
with zipfile.ZipFile('/kaggle/working/xai_boosting.zip', 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in pathlib.Path('data/results').rglob('*.csv'):
        if f.name.startswith(keep):
            zf.write(f, str(f))
    for f in [*pathlib.Path('data/results').rglob('figures/xai-*.png'), *pathlib.Path('logs').rglob('*.log')]:
        zf.write(f, str(f))
print(round(pathlib.Path('/kaggle/working/xai_boosting.zip').stat().st_size / 1e6, 2), 'MB')
shutil.rmtree('/kaggle/working/capstone', ignore_errors=True)  # keep the Output to the zip
notify('Boosting + XAI: FINISHED' if ok else 'Boosting + XAI: finished WITH ERRORS',
       'xai_boosting.zip is in the Output tab', 'default' if ok else 'high', 'tada' if ok else 'warning')